# Basic Catia .CATPart Extraction Example

This is an example of how a user would perform the following steps:
- Update the user's function access to include the Catia module
- Import the istari-digital-client from PyPI
- Instantiate an instance of the istari-digital-client Client class
- Upload a Catia model
- Extract the Catia model
- View the extracted artifacts

## Update function access

The Istari administrator must add the user to the relevant function on the Function Access page. This can be accessed in the Istari environment via Admin Panel -> Function Access -> Relevant Function -> Manage Function Access. Search to find the appropriate user, click add at the top of the window, then click save at the bottom of the window. 

## Install dependencies from PyPI

#### Note: the pip command below installs the most recent version of the digital client. Depending on your Istari release, you may need to install an older version. Refer to your relevant release page in the [release docs](https://docs.istaridigital.com/releases/2025-06-01-Release) for the appropriate SDK client version. The commented line below shows the command to install a specific client version. 

In [1]:
!pip install istari-digital-client
# !pip install istari-digital-client==7.4.4 # use this command to install the appropriate client version if you are on an older Istari release 

!pip install python-dotenv

In [2]:
import istari_digital_client as istari_digital
import os
from dotenv import load_dotenv
from pathlib import Path
import time
from datetime import datetime

## Instantiate the istari-digital-client Client class

To interact with Istari Digital, we need to create an instance of the istari-digital-client Client class.

The Client class takes a Configuration object that contains the following parameters:
- registry_url (required): The URL of the Istari Digital Registry Service
- registry_auth_token (required): The authentication token to use to authenticate with the Istari Digital Registry Service
- retry_enabled (optional): Whether to retry failed requests.  Defaults to True
- retry_max_attempts (optional): The maximum number of retry attempts.  Defaults to 3
- retry_min_interval_millis (optional): The minimum interval between retry attempts in milliseconds.
- retry_max_interval_millis (optional): The maximum interval between retry attempts in milliseconds.
- retry_jitter_enabled (optional): Whether to use jitter when retrying failed requests.  Defaults to True
- filesystem_cache_enabled (optional): Whether to use the filesystem cache.  Defaults to True
- filesystem_cache_root (optional): The root directory of the filesystem cache.
- filesystem_cache_clean_on_exit (optional): Whether to clean the filesystem cache on exit.  Defaults to True
- multipart_chunksize (optional): The chunk size to use when uploading files.
- multipart_threshold (optional): The threshold size to use when uploading files.

We MUST set the registry_url and registry_auth_token parameters to interact with Istari Digital.
In this example, we are following best practices and using environment variables to store the
registry_url and registry_auth_token. The environment variables are loaded using the python-dotenv package.

The configuration parameters are then used to instantiate the Configuration class.
Once instantiated, the configuration object is passed to the istari-digital-client Client class to create
an instance of the Client class.

In [3]:
load_dotenv()

dev_auth_token = os.getenv("REGISTRY_ACCESS_TOKEN")
assert dev_auth_token is not None

dev_registry_url = os.getenv("REGISTRY_URL")
assert dev_registry_url is not None

configuration = istari_digital.Configuration(
    registry_url=dev_registry_url,
    registry_auth_token=dev_auth_token,
)
assert configuration is not None

client = istari_digital.Client(
    config = configuration
)
assert client is not None

2025-07-25 15:14:53 - istari-digital-client - INFO - Logging configured with level: INFO


## Upload the Catia .CATPart file to the Istari Digital Registry Service

Before extracting the Catia .CATPart file, the file must be uploaded to the Istari Digital Registry Service.
To do this, we use the add_model method of the client object.
The add_model method takes the following parameters:
- path: The path to the file to upload.
- description: An optional description of the file
- version_name: An optional version name of the file
- external_identifier: An optional external identifier of the file
- display_name: An optional display name of the file.  Useful for displaying in the UI

The parameters are then passed to the add_model method of the client object to upload the file.
The add_model method returns a Model object that contains the metadata of the uploaded file.

We can then validate that the file was successfully uploaded by accessing the properties of
the Model object that is returned.

In [4]:
base_path = Path.cwd()
file_path = base_path / "files/WING.CATPart"

description = "Catia model v1"
version_name = "v1"
external_identifier = "catia_model_v1"
display_name = "Catia Model v1"

catia_model = client.add_model(
    path=file_path,
    description=description,
    version_name=version_name,
    external_identifier=external_identifier,
    display_name=display_name,
)

assert catia_model is not None
assert isinstance(catia_model, istari_digital.Model)

assert catia_model.description == description
assert catia_model.version_name == version_name
assert catia_model.external_identifier == external_identifier
assert catia_model.display_name == display_name
assert catia_model.read_bytes() == file_path.read_bytes()

## Extract the uploaded Catia .CATPart file

To extract the uploaded Catia .CATPart file, the following params are needed:
- model_id: The id of the model to extract
- function: The extraction function to run
- tool_name: The tool that is needed to extract the Catia file
- tool_version: The version of the tool
- operating_system: The operating system that the tool will run on

The extract parameters are then passed to the add_job method of the client object to begin the extraction job.

Once the job is created, we poll the job until it is completed.  The job is polled by checking the JobStatusName of the job.

In [6]:
model_id = catia_model.id
function = "@istari:extract"
tool_name = "dassault_catia_v5"
tool_version = "6R2023"
operating_system = "Windows 10"

job = client.add_job(
    model_id=model_id,
    function=function,
    tool_name=tool_name,
    tool_version=tool_version,
    operating_system=operating_system,
)

start_time = datetime.now()
print(f"Extraction started for model ID {model_id}, job ID: {job.id}")

assert job is not None
assert isinstance(job, istari_digital.Job)

while job.status.name not in [istari_digital.JobStatusName.COMPLETED, istari_digital.JobStatusName.FAILED]:
    elapsed_time = datetime.now()-start_time
    print(f"\rExtraction job {job.id} status: {job.status.name.value} ({elapsed_time})", end="")
    time.sleep(5)
    job = client.get_job(job.id)

if job.status.name.value == "Completed":
    print(f"\r\rExtraction job {job.id} completed successfully!                         ")
    model = client.get_model(job.model.id)
    print(f"\nThe following artifacts were extracted:")
    for artifact in model.artifacts:
        print(f"- artifact id: {artifact.id}")
        print(f"  revision id: {artifact.revision.id}")
        print(f"  extension: {artifact.extension}")
        print(f"  mime type: {artifact.mime}")
        print(f"  name: {artifact.name}")
else:
    print(f"\r\rExtraction job {job.id} failed with status: {job.status.name.value}")

assert job.status.name == istari_digital.JobStatusName.COMPLETED

Extraction started for model ID 9e91c54c-bccb-4444-8f78-65c9f79679e1, job ID: 0e83c610-7655-474e-b2d1-0c80f1b58a42
Extraction job 0e83c610-7655-474e-b2d1-0c80f1b58a42 completed successfully!                         

The following artifacts were extracted:
- artifact id: a0f0409e-b20c-47ef-9491-2e41e7673f35
  revision id: 9fe7622b-05da-4950-9ea2-1663aa829e79
  extension: obj
  mime type: None
  name: geometry.obj
- artifact id: 76e55d1e-9829-4aab-95b4-d10236bf3484
  revision id: 52658a03-8bf3-4407-bad0-c300dc6ec580
  extension: json
  mime type: None
  name: bill_of_materials.json
- artifact id: 3fbde5a8-b2d0-408f-be3b-ac93ff0278f0
  revision id: 8ea60b8f-ccc4-450c-84ac-7dbd71525692
  extension: json
  mime type: None
  name: parameters.json
- artifact id: 64f7cb1b-6d12-44c4-a34b-87c7b31c1aa6
  revision id: 8d7b7fda-1fb6-4a4c-b99f-debbcb9045de
  extension: json
  mime type: None
  name: parts.json
- artifact id: 3d41f67d-9a8b-4af0-8620-1d57e28209ac
  revision id: 8383666e-d56d-4d3c-bdf

## Clean up the uploaded file

The uploaded file can be archived using the archive_model method of the client object.
The archive_model method takes the following parameters:
- model_id: The id of the model to archive
- archive: An Archive object that contains the reason for archiving the model

This is useful for cleaning up the files if you are going to do multiple runs of the notebook.

In [7]:
archive_reason = istari_digital.Archive(
    reason="This file was used for an example"
)

archived_model = client.archive_model(
    model_id=model_id,
    archive=archive_reason,
)

assert archived_model is not None
assert archived_model.archive_status.name == istari_digital.ArchiveStatusName.ARCHIVED